In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split,StratifiedKFold,cross_val_score,RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, confusion_matrix

In [2]:
df=pd.read_csv('..\DATASET\stage2_train_resampled.csv')
val=pd.read_csv('..\DATASET\stage2_val_clean.csv')
test=pd.read_csv('..\DATASET\stage2_test_clean.csv')

<>:1: SyntaxWarning: invalid escape sequence '\D'
<>:2: SyntaxWarning: invalid escape sequence '\D'
<>:3: SyntaxWarning: invalid escape sequence '\D'
<>:1: SyntaxWarning: invalid escape sequence '\D'
<>:2: SyntaxWarning: invalid escape sequence '\D'
<>:3: SyntaxWarning: invalid escape sequence '\D'
C:\Users\dyash\AppData\Local\Temp\ipykernel_25068\2797984614.py:1: SyntaxWarning: invalid escape sequence '\D'
  df=pd.read_csv('..\DATASET\stage2_train_resampled.csv')
C:\Users\dyash\AppData\Local\Temp\ipykernel_25068\2797984614.py:2: SyntaxWarning: invalid escape sequence '\D'
  val=pd.read_csv('..\DATASET\stage2_val_clean.csv')
C:\Users\dyash\AppData\Local\Temp\ipykernel_25068\2797984614.py:3: SyntaxWarning: invalid escape sequence '\D'
  test=pd.read_csv('..\DATASET\stage2_test_clean.csv')


In [3]:
df.head()

,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,80,4004,6,0,6,0,6,0,1.000000,2.449490,...,20,0.0,0.0,0,0,0.0,0.0,0,0,1
1,80,190206,3,6,26,11607,20,0,8.666667,10.263203,...,20,0.0,0.0,0,0,0.0,0.0,0,0,1
2,80,1000000,6,7,333,11595,333,0,55.500000,135.946681,...,32,482.0,0.0,482,482,1000000.0,0.0,1000000,1000000,1
3,80,1000000,7,7,368,11595,368,0,52.571429,139.090926,...,32,6.0,0.0,6,6,1000000.0,0.0,1000000,1000000,1
4,80,1000000,6,8,423,11595,405,0,70.500000,163.897224,...,20,11018.0,0.0,11018,11018,1000000.0,0.0,1000000,1000000,1


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1042512 entries, 0 to 1042511
Data columns (total 79 columns):
 #   Column                       Non-Null Count    Dtype  
---  ------                       --------------    -----  
 0   Destination Port             1042512 non-null  int64  
 1   Flow Duration                1042512 non-null  int64  
 2   Total Fwd Packets            1042512 non-null  int64  
 3   Total Backward Packets       1042512 non-null  int64  
 4   Total Length of Fwd Packets  1042512 non-null  int64  
 5   Total Length of Bwd Packets  1042512 non-null  int64  
 6   Fwd Packet Length Max        1042512 non-null  int64  
 7   Fwd Packet Length Min        1042512 non-null  int64  
 8   Fwd Packet Length Mean       1042512 non-null  float64
 9   Fwd Packet Length Std        1042512 non-null  float64
 10  Bwd Packet Length Max        1042512 non-null  int64  
 11  Bwd Packet Length Min        1042512 non-null  int64  
 12  Bwd Packet Length Mean       1042512 non-n

In [5]:
test.shape

(42588, 79)

In [6]:
df['Label'].value_counts()

Label
1    260628
2    260628
3    260628
4    260628
Name: count, dtype: int64

In [7]:
df.shape

(1042512, 79)

In [8]:
val.shape

(38321, 79)

In [9]:

X_train = df.drop(columns=['Label'])
y_train = df['Label']

X_val = val.drop(columns=['Label'])
y_val = val['Label']

X_test = test.drop(columns=['Label'])
y_test = test['Label']

# Check
print("Train:", X_train.shape)
print("Val  :", X_val.shape)
print("Test :", X_test.shape)

Train: (1042512, 78)
Val  : (38321, 78)
Test : (42588, 78)


In [22]:

X_train_round = X_train.round(5)
X_val_round   = X_val.round(5)


train_set = set(map(tuple, X_train_round.values))


mask = ~X_val_round.apply(tuple, axis=1).isin(train_set)


X_val = X_val[mask].reset_index(drop=True)
y_val = y_val[mask].reset_index(drop=True)

print("New val size:", X_val.shape)

AttributeError: 'numpy.ndarray' object has no attribute 'reset_index'

In [11]:
train_set = set(map(tuple, X_train_round.values))
val_set   = set(map(tuple, X_val.round(5).values))

overlap = train_set & val_set
print("Overlap after fix:", len(overlap))

Overlap after fix: 0


In [12]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

y_train = le.fit_transform(y_train)
y_val   = le.transform(y_val)
y_test  = le.transform(y_test)

In [13]:
np.unique(y_test)

array([0, 1, 2, 3])

In [14]:
from xgboost import XGBClassifier
from sklearn.metrics import f1_score
from xgboost.callback import EarlyStopping

best_score = 0
best_params = None

for depth in [3, 4, 5]:
    for lr in [0.03, 0.05]:
        for n in [200, 300]:

            model = XGBClassifier(
                objective='multi:softprob',
                num_class=4,
                max_depth=depth,
                learning_rate=lr,
                n_estimators=n,
                subsample=0.8,
                colsample_bytree=0.8,
                eval_metric='mlogloss',
                random_state=42,
                n_jobs=-1
            )

            model.fit(
                X_train,
                y_train,
                eval_set=[(X_val, y_val)],
                
            )

            y_pred = model.predict(X_val)
            score = f1_score(y_val, y_pred, average='macro')

            if score > best_score:
                best_score = score
                best_params = (depth, lr, n)

print("Best params:", best_params)
print("Best val F1:", best_score)

[0]	validation_0-mlogloss:1.32940
[1]	validation_0-mlogloss:1.27508
[2]	validation_0-mlogloss:1.22453
[3]	validation_0-mlogloss:1.17754
[4]	validation_0-mlogloss:1.13212
[5]	validation_0-mlogloss:1.08992
[6]	validation_0-mlogloss:1.04907
[7]	validation_0-mlogloss:1.01044
[8]	validation_0-mlogloss:0.97371
[9]	validation_0-mlogloss:0.93908
[10]	validation_0-mlogloss:0.90590
[11]	validation_0-mlogloss:0.87449
[12]	validation_0-mlogloss:0.84459
[13]	validation_0-mlogloss:0.81614
[14]	validation_0-mlogloss:0.78836
[15]	validation_0-mlogloss:0.76187
[16]	validation_0-mlogloss:0.73615
[17]	validation_0-mlogloss:0.71154
[18]	validation_0-mlogloss:0.68803
[19]	validation_0-mlogloss:0.66537
[20]	validation_0-mlogloss:0.64357
[21]	validation_0-mlogloss:0.62289
[22]	validation_0-mlogloss:0.60282
[23]	validation_0-mlogloss:0.58346
[24]	validation_0-mlogloss:0.56520
[25]	validation_0-mlogloss:0.54734
[26]	validation_0-mlogloss:0.52998
[27]	validation_0-mlogloss:0.51332
[28]	validation_0-mlogloss:0.4

In [15]:
# model = XGBClassifier(
#     **rs.best_params_,
#     objective='multi:softprob',
#     num_class=4,
#     tree_method='hist',
#     eval_metric='mlogloss',
#     use_label_encoder=False,
#     early_stopping_rounds=20,
#     n_jobs=-1,
#     random_state=42
# )

# model.fit(
#     X_train, y_train,
#     eval_set=[(X_val,y_val)],
#     verbose=50
# )

# print('\nTraining complete.')
# print(f'Best iteration: {model.best_iteration}')


In [16]:
y_pred=model.predict(X_test)

In [17]:
print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00     32177
           1       1.00      1.00      1.00      9082
           2       1.00      1.00      1.00       915
           3       0.99      1.00      1.00       414

    accuracy                           1.00     42588
   macro avg       1.00      1.00      1.00     42588
weighted avg       1.00      1.00      1.00     42588



In [18]:
y_pred = model.predict(X_val)

print("Prediction distribution:\n", pd.Series(y_pred).value_counts())
print("\nActual distribution:\n", pd.Series(y_val).value_counts())

Prediction distribution:
 0    28734
1     6948
2      824
3      355
Name: count, dtype: int64

Actual distribution:
 0    28741
1     6943
2      824
3      353
Name: count, dtype: int64


In [19]:
y_test_pred = model.predict(X_test)

print("Test distribution:\n", pd.Series(y_test_pred).value_counts())
print("\nTest actual:\n", pd.Series(y_test).value_counts())

from sklearn.metrics import f1_score
print("\nTest F1:", f1_score(y_test, y_test_pred, average='macro'))

Test distribution:
 0    32171
1     9085
2      914
3      418
Name: count, dtype: int64

Test actual:
 0    32177
1     9082
2      915
3      414
Name: count, dtype: int64

Test F1: 0.9985615031349944


In [21]:
import joblib
from pathlib import Path
models_dir = Path('../MODELS')
models_dir.mkdir(parents=True, exist_ok=True)
# Save under the user-requested name and also the UI-expected filename
joblib.dump(model, models_dir / 'xg_boost_stage2.pkl')
joblib.dump(model, models_dir / 'xgboost_stage2.pkl')
print('Saved models:', models_dir / 'xg_boost_stage2.pkl', models_dir / 'xgboost_stage2.pkl')

Saved models: ..\MODELS\xg_boost_stage2.pkl ..\MODELS\xgboost_stage2.pkl
